## EDA

In [1]:
import pandas as pd

df = pd.read_csv('devset_videolist_GT.csv')
print(df['channelName'].value_counts())
print(df[['memorability_score','brand_memorability','durationSeconds','engagementRate']].describe())
print(df['memorability_score'].corr(df['brand_memorability']))

channelName
Goldman Sachs                                   79
UBS                                             37
jpmorgan                                        31
Aon                                             18
Credit Suisse                                   17
Vanguard                                        17
Allianz                                         14
Legal & General Investment Management - LGIM    14
HSBC UK                                         12
Janus Henderson Investment Trusts               11
Legal & General                                 10
BMO Global Asset Management - EMEA               8
Invesco                                          8
Amundi                                           7
Blackstone                                       7
BNP Paribas                                      6
Aviva                                            5
Life at Capital Group                            4
Aon Assessment Solutions                         4
T. Rowe Price      

In [2]:
import numpy as np

vit_feature_sample = np.load('features/ViT/_1qVUdvVC9A.npy')
r3d_feature_sample = np.load('features/R3D/_1qVUdvVC9A.npy')
histogram_feature_sample = np.load('features/RGBHistogram/_1qVUdvVC9A.npy')

print('ViT feature sample shape:', vit_feature_sample.shape)
print('R3D feature sample shape:', r3d_feature_sample.shape)
print('Histogram feature sample shape:', histogram_feature_sample.shape)

ViT feature sample shape: (3, 768)
R3D feature sample shape: (512,)
Histogram feature sample shape: (3, 24)


## Feature Engineering

In [3]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

# ── 1. Load CSV ──────────────────────────────────────────────────────────────
df = pd.read_csv('devset_videolist_GT.csv')

# ── 2. Metadata features ─────────────────────────────────────────────────────
import numpy as np

df['log_duration']      = np.log1p(df['durationSeconds'])
df['log_views']         = np.log1p(df['viewsCount'])
df['log_likes']         = np.log1p(df['likesCount'])
df['log_engagement']    = np.log1p(df['engagementRate'])
df['has_tags']          = (df['tags'].notna() & (df['tags'] != '')).astype(int)
df['has_description']   = (df['description'].notna()).astype(int)

metadata_cols = ['log_duration', 'log_views', 'log_likes',
                 'log_engagement', 'categoryId', 'has_tags', 'has_description']

# ── 3. Load & aggregate visual features ──────────────────────────────────────
FEATURE_DIR = Path('features')
FEATURE_NAMES = ['ViT', 'R3D', 'ResNet50', 'VGG', 'AlexNet',
                 'DenseNet121', 'EfficientNetB3',
                 'RGBHistogram', 'HSVHistogram', 'LBP']

def load_feature(video_id, feature_name):
    path = FEATURE_DIR / feature_name / f'{video_id}.npy'
    if not path.exists():
        return None
    feat = np.load(path)
    # If frame-level (2D), mean pool across frames
    if feat.ndim == 2:
        feat = feat.mean(axis=0)
    return feat

# Build feature matrix for each modality
feature_matrices = {}
missing_report = {}

for feat_name in FEATURE_NAMES:
    vecs = []
    missing = []
    for vid_id in df['id']:
        # Note: filenames have underscore prefix for some - check both
        vec = load_feature(vid_id, feat_name)
        if vec is None:
            vec = load_feature(f'_{vid_id}', feat_name)
        if vec is None:
            missing.append(vid_id)
            vecs.append(None)
        else:
            vecs.append(vec)
    
    missing_report[feat_name] = len(missing)
    
    # Fill missing with zeros (or mean later)
    dim = next(v for v in vecs if v is not None).shape[0]
    matrix = np.array([v if v is not None else np.zeros(dim) for v in vecs])
    feature_matrices[feat_name] = matrix
    print(f'{feat_name}: shape={matrix.shape}, missing={len(missing)}')

print("\nMissing files per feature:", missing_report)

ViT: shape=(339, 768), missing=0
R3D: shape=(339, 512), missing=0
ResNet50: shape=(339, 2048), missing=0
VGG: shape=(339, 4096), missing=0
AlexNet: shape=(339, 4096), missing=0
DenseNet121: shape=(339, 1024), missing=0
EfficientNetB3: shape=(339, 1536), missing=0
RGBHistogram: shape=(339, 24), missing=0
HSVHistogram: shape=(339, 24), missing=0
LBP: shape=(339, 256), missing=0

Missing files per feature: {'ViT': 0, 'R3D': 0, 'ResNet50': 0, 'VGG': 0, 'AlexNet': 0, 'DenseNet121': 0, 'EfficientNetB3': 0, 'RGBHistogram': 0, 'HSVHistogram': 0, 'LBP': 0}


## Baseline Model with Brand-Isolated Cross-Validation

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# ── 0. Reload everything ─────────────────────────────────────────────────────
df = pd.read_csv('devset_videolist_GT.csv')
df['log_duration']   = np.log1p(df['durationSeconds'])
df['log_views']      = np.log1p(df['viewsCount'])
df['log_likes']      = np.log1p(df['likesCount'])
df['log_engagement'] = np.log1p(df['engagementRate'])
df['has_tags']       = (df['tags'].notna() & (df['tags'] != '')).astype(int)

FEATURE_DIR = Path('features')
FEATURE_NAMES = ['ViT','R3D','ResNet50','VGG','AlexNet',
                 'DenseNet121','EfficientNetB3',
                 'RGBHistogram','HSVHistogram','LBP']

def load_feature(video_id, feature_name):
    for vid in [video_id, f'_{video_id}']:
        path = FEATURE_DIR / feature_name / f'{vid}.npy'
        if path.exists():
            feat = np.load(path)
            return feat.mean(axis=0) if feat.ndim == 2 else feat
    return None

feature_matrices = {}
for feat_name in FEATURE_NAMES:
    vecs = [load_feature(vid, feat_name) for vid in df['id']]
    dim = next(v for v in vecs if v is not None).shape[0]
    feature_matrices[feat_name] = np.array(
        [v if v is not None else np.zeros(dim) for v in vecs]
    )

# ── 1. Brand-Isolated Cross-Validation Folds ─────────────────────────────────
# Goldman Sachs (79 videos) gets its own fold due to size
# Other channels grouped into ~4 balanced folds

channel_counts = df['channelName'].value_counts()
print("Building brand-isolated folds...")
print(channel_counts.to_string())

# Assign each channel to a fold
# Fold 0: Goldman Sachs alone (79 videos, 23%)
# Folds 1-4: remaining channels grouped by size (greedy bin packing)

N_FOLDS = 5
fold_assignments = {}  # channel -> fold_id

# Goldman Sachs gets fold 0
fold_assignments['Goldman Sachs'] = 0
fold_sizes = {0: 79, 1: 0, 2: 0, 3: 0, 4: 0}

# Assign remaining channels greedily to smallest fold (1-4)
remaining = channel_counts.drop('Goldman Sachs')
for channel, count in remaining.items():
    # Pick fold with smallest current size among folds 1-4
    target_fold = min([1, 2, 3, 4], key=lambda f: fold_sizes[f])
    fold_assignments[channel] = target_fold
    fold_sizes[target_fold] += count

# Map each video to its fold
df['fold'] = df['channelName'].map(fold_assignments)
print("\nFold sizes (videos):")
print(df['fold'].value_counts().sort_index())
print("\nChannels per fold:")
for f in range(N_FOLDS):
    channels = [ch for ch, fold in fold_assignments.items() if fold == f]
    print(f"  Fold {f}: {channels}")


# ── 2. Evaluation Function ────────────────────────────────────────────────────
def cross_val_spearman(X, y_mem, y_brand, df, n_components=None, alpha=10.0):
    """
    Brand-isolated CV. Returns mean Spearman r for both targets.
    Optionally applies PCA before Ridge.
    """
    preds_mem   = np.zeros(len(df))
    preds_brand = np.zeros(len(df))

    for fold_id in range(N_FOLDS):
        val_mask   = df['fold'] == fold_id
        train_mask = ~val_mask

        X_train, X_val = X[train_mask], X[val_mask]
        ym_train = y_mem[train_mask]
        yb_train = y_brand[train_mask]

        # Scale
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val   = scaler.transform(X_val)

        # Optional PCA
        if n_components is not None:
            n_comp = min(n_components, X_train.shape[1], X_train.shape[0] - 1)
            pca = PCA(n_components=n_comp)
            X_train = pca.fit_transform(X_train)
            X_val   = pca.transform(X_val)

        # Fit Ridge for memorability
        ridge_mem = Ridge(alpha=alpha)
        ridge_mem.fit(X_train, ym_train)
        preds_mem[val_mask] = ridge_mem.predict(X_val)

        # Fit Ridge for brand memorability
        ridge_brand = Ridge(alpha=alpha)
        ridge_brand.fit(X_train, yb_train)
        preds_brand[val_mask] = ridge_brand.predict(X_val)

    spear_mem   = spearmanr(y_mem,   preds_mem).correlation
    spear_brand = spearmanr(y_brand, preds_brand).correlation
    return spear_mem, spear_brand


y_mem   = df['memorability_score'].values
y_brand = df['brand_memorability'].values


# ── 3. Experiment 1: Metadata-only Baseline ───────────────────────────────────
metadata_cols = ['log_duration','log_views','log_likes',
                 'log_engagement','categoryId','has_tags']
X_meta = df[metadata_cols].fillna(0).values

r_mem, r_brand = cross_val_spearman(X_meta, y_mem, y_brand, df,
                                     n_components=None, alpha=10.0)
print(f"\n{'='*50}")
print(f"Metadata-only baseline")
print(f"  Memorability Spearman r:       {r_mem:.4f}")
print(f"  Brand Memorability Spearman r: {r_brand:.4f}")


# ── 4. Experiment 2: Each visual feature individually ────────────────────────
print(f"\n{'='*50}")
print(f"Single-feature results (PCA→100 + Ridge):")
print(f"{'Feature':<20} {'Mem r':>8} {'Brand r':>8}")
print("-" * 40)

single_results = {}
for feat_name in FEATURE_NAMES:
    X = feature_matrices[feat_name]
    r_mem, r_brand = cross_val_spearman(X, y_mem, y_brand, df,
                                         n_components=100, alpha=10.0)
    single_results[feat_name] = (r_mem, r_brand)
    print(f"{feat_name:<20} {r_mem:>8.4f} {r_brand:>8.4f}")


# ── 5. Experiment 3: All features concatenated ───────────────────────────────
X_all_visual = np.concatenate(list(feature_matrices.values()), axis=1)
r_mem, r_brand = cross_val_spearman(X_all_visual, y_mem, y_brand, df,
                                     n_components=150, alpha=10.0)
print(f"\n{'='*50}")
print(f"All visual features concatenated (PCA→150 + Ridge)")
print(f"  Memorability Spearman r:       {r_mem:.4f}")
print(f"  Brand Memorability Spearman r: {r_brand:.4f}")


# ── 6. Experiment 4: Best visual + metadata ──────────────────────────────────
# Find best single visual feature for each target
best_mem_feat   = max(single_results, key=lambda k: single_results[k][0])
best_brand_feat = max(single_results, key=lambda k: single_results[k][1])
print(f"\nBest single feature for memorability: {best_mem_feat}")
print(f"Best single feature for brand:        {best_brand_feat}")

scaler_meta = StandardScaler()
X_meta_scaled = scaler_meta.fit_transform(df[metadata_cols].fillna(0).values)

X_best_mem   = np.concatenate([feature_matrices[best_mem_feat], X_meta_scaled], axis=1)
X_best_brand = np.concatenate([feature_matrices[best_brand_feat], X_meta_scaled], axis=1)

r_mem, _       = cross_val_spearman(X_best_mem,   y_mem,   y_brand, df, n_components=100, alpha=10.0)
_, r_brand     = cross_val_spearman(X_best_brand, y_mem,   y_brand, df, n_components=100, alpha=10.0)
print(f"\nBest visual + metadata:")
print(f"  Memorability Spearman r:       {r_mem:.4f}")
print(f"  Brand Memorability Spearman r: {r_brand:.4f}")

Building brand-isolated folds...
channelName
Goldman Sachs                                   79
UBS                                             37
jpmorgan                                        31
Aon                                             18
Credit Suisse                                   17
Vanguard                                        17
Allianz                                         14
Legal & General Investment Management - LGIM    14
HSBC UK                                         12
Janus Henderson Investment Trusts               11
Legal & General                                 10
BMO Global Asset Management - EMEA               8
Invesco                                          8
Amundi                                           7
Blackstone                                       7
BNP Paribas                                      6
Aviva                                            5
Life at Capital Group                            4
Aon Assessment Solutions             

## Diagnosis: Near-Zero / Negative Results

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

# ── Assume df, feature_matrices, y_mem, y_brand already loaded ───────────────

# ── DIAGNOSTIC 1: Random CV vs Brand-Isolated CV ─────────────────────────────
# If random CV gives much higher scores, the brand visual style
# is dominating the signal (overfitting to brand identity)

def random_cv_spearman(X, y, alpha=10.0, n_components=100, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    preds = np.zeros(len(y))
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr = y[train_idx]
        scaler = StandardScaler()
        X_tr  = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)
        n_comp = min(n_components, X_tr.shape[1], X_tr.shape[0]-1)
        pca = PCA(n_components=n_comp)
        X_tr  = pca.fit_transform(X_tr)
        X_val = pca.transform(X_val)
        ridge = Ridge(alpha=alpha)
        ridge.fit(X_tr, y_tr)
        preds[val_idx] = ridge.predict(X_val)
    return spearmanr(y, preds).correlation

print("="*60)
print("DIAGNOSTIC 1: Random CV vs Brand-Isolated CV")
print("(Big gap = model learns brand style, not memorability)")
print(f"{'Feature':<20} {'RandCV_Mem':>10} {'BrandCV_Mem':>12}")
print("-"*45)
for feat_name in ['ViT', 'R3D', 'ResNet50', 'LBP', 'RGBHistogram']:
    X = feature_matrices[feat_name]
    r_rand  = random_cv_spearman(X, y_mem, n_components=100)
    r_brand = cross_val_spearman(X, y_mem, y_brand, df,
                                  n_components=100, alpha=10.0)[0]
    print(f"{feat_name:<20} {r_rand:>10.4f} {r_brand:>12.4f}")


# ── DIAGNOSTIC 2: Per-fold breakdown ─────────────────────────────────────────
# Which folds are hurting? Goldman Sachs fold (0) may be dragging scores down
print("\n" + "="*60)
print("DIAGNOSTIC 2: Per-fold Spearman (ViT feature)")

X = feature_matrices['ViT']
for fold_id in range(5):
    val_mask   = df['fold'] == fold_id
    train_mask = ~val_mask
    X_tr = StandardScaler().fit_transform(X[train_mask])
    X_val_raw = X[val_mask]
    scaler = StandardScaler()
    X_tr  = scaler.fit_transform(X[train_mask])
    X_val = scaler.transform(X_val_raw)
    pca = PCA(n_components=100)
    X_tr  = pca.fit_transform(X_tr)
    X_val = pca.transform(X_val)
    ridge = Ridge(alpha=10.0)
    ridge.fit(X_tr, y_mem[train_mask])
    preds = ridge.predict(X_val)
    r = spearmanr(y_mem[val_mask], preds).correlation
    channels = df[val_mask]['channelName'].unique().tolist()
    print(f"  Fold {fold_id} (n={val_mask.sum():3d}): r={r:.4f}  {channels[:2]}...")


# ── DIAGNOSTIC 3: Alpha sweep ────────────────────────────────────────────────
print("\n" + "="*60)
print("DIAGNOSTIC 3: Alpha sweep (ViT, brand-isolated CV)")
print(f"{'Alpha':>10} {'Mem r':>8} {'Brand r':>8}")
for alpha in [0.01, 0.1, 1, 10, 100, 1000, 10000]:
    X = feature_matrices['ViT']
    r_m, r_b = cross_val_spearman(X, y_mem, y_brand, df,
                                   n_components=100, alpha=alpha)
    print(f"{alpha:>10.2f} {r_m:>8.4f} {r_b:>8.4f}")


# ── DIAGNOSTIC 4: PCA components sweep ───────────────────────────────────────
print("\n" + "="*60)
print("DIAGNOSTIC 4: PCA components sweep (ViT)")
print(f"{'n_components':>14} {'Mem r':>8} {'Brand r':>8}")
for n_comp in [10, 20, 50, 100, 150, 200, None]:
    X = feature_matrices['ViT']
    r_m, r_b = cross_val_spearman(X, y_mem, y_brand, df,
                                   n_components=n_comp, alpha=100)
    label = str(n_comp) if n_comp else 'None(full)'
    print(f"{label:>14} {r_m:>8.4f} {r_b:>8.4f}")


# ── DIAGNOSTIC 5: SVR instead of Ridge ───────────────────────────────────────
print("\n" + "="*60)
print("DIAGNOSTIC 5: SVR with RBF kernel (ViT, PCA=50)")

def svr_cv_spearman(X, y_mem, y_brand, df, n_components=50):
    preds_mem   = np.zeros(len(df))
    preds_brand = np.zeros(len(df))
    for fold_id in range(5):
        val_mask   = df['fold'] == fold_id
        train_mask = ~val_mask
        scaler = StandardScaler()
        X_tr  = scaler.fit_transform(X[train_mask])
        X_val = scaler.transform(X[val_mask])
        n_comp = min(n_components, X_tr.shape[1], X_tr.shape[0]-1)
        pca = PCA(n_components=n_comp)
        X_tr  = pca.fit_transform(X_tr)
        X_val = pca.transform(X_val)
        for y, preds in [(y_mem, preds_mem), (y_brand, preds_brand)]:
            svr = SVR(kernel='rbf', C=1.0, epsilon=0.05)
            svr.fit(X_tr, y[train_mask])
            preds[val_mask] = svr.predict(X_val)
    return (spearmanr(y_mem, preds_mem).correlation,
            spearmanr(y_brand, preds_brand).correlation)

for feat_name in ['ViT', 'R3D', 'LBP']:
    X = feature_matrices[feat_name]
    r_m, r_b = svr_cv_spearman(X, y_mem, y_brand, df, n_components=50)
    print(f"  {feat_name:<15} Mem={r_m:.4f}  Brand={r_b:.4f}")

DIAGNOSTIC 1: Random CV vs Brand-Isolated CV
(Big gap = model learns brand style, not memorability)
Feature              RandCV_Mem  BrandCV_Mem
---------------------------------------------
ViT                      0.0285      -0.0299
R3D                      0.0272       0.0072
ResNet50                -0.0043      -0.0445
LBP                      0.0462      -0.0237
RGBHistogram             0.0888      -0.0262

DIAGNOSTIC 2: Per-fold Spearman (ViT feature)
  Fold 0 (n= 79): r=-0.0001  ['Goldman Sachs']...
  Fold 1 (n= 65): r=-0.0716  ['Legal & General', 'UBS']...
  Fold 2 (n= 65): r=0.0791  ['jpmorgan', 'Legal & General Investment Management - LGIM']...
  Fold 3 (n= 65): r=0.0472  ['Allianz', 'BNP Paribas']...
  Fold 4 (n= 65): r=0.0282  ['Aviva', 'Amundi']...

DIAGNOSTIC 3: Alpha sweep (ViT, brand-isolated CV)
     Alpha    Mem r  Brand r
      0.01  -0.0278  -0.0285
      0.10  -0.0415  -0.0225
      1.00  -0.0491  -0.0333
     10.00  -0.0497  -0.0410
    100.00  -0.0575  -0.0518
 

## STT Text Features

In [6]:
import numpy as np
import pandas as pd
import os, re
from pathlib import Path
from scipy.stats import spearmanr, pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

STT_DIR = Path('devset-stt')

# ── 1. Load all transcripts ───────────────────────────────────────────────────
transcripts = {}
missing_stt = []

for vid_id in df['id']:
    found = False
    for name in [vid_id, f'_{vid_id}']:
        path = STT_DIR / f'{name}.txt'
        if path.exists():
            transcripts[vid_id] = path.read_text(encoding='utf-8').strip()
            found = True
            break
    if not found:
        transcripts[vid_id] = ''
        missing_stt.append(vid_id)

print(f"Missing STT files: {len(missing_stt)}")
print(f"Empty transcripts: {sum(1 for t in transcripts.values() if t == '')}")

# Sample stats
lengths = [len(t.split()) for t in transcripts.values()]
print(f"Transcript word count — min:{min(lengths)}, "
      f"max:{max(lengths)}, mean:{np.mean(lengths):.0f}, "
      f"median:{np.median(lengths):.0f}")


# ── 2. Hand-crafted text features ────────────────────────────────────────────
# These are interpretable signals specific to memorability in commercials

def extract_text_features(vid_id, channel_name):
    text = transcripts.get(vid_id, '').lower()
    words = text.split()
    n_words = len(words)

    # Basic length features
    n_chars    = len(text)
    n_sentences = text.count('.') + text.count('!') + text.count('?') + 1
    avg_sent_len = n_words / max(n_sentences, 1)

    # Brand mention frequency — does the brand name appear in the speech?
    brand_keywords = channel_name.lower().split()
    brand_keywords = [w for w in brand_keywords
                      if w not in {'&','the','of','at','group','global',
                                   'asset','management','investment','uk',
                                   'us','emea','apac','solutions','managers',
                                   'life','partners','general'}]
    brand_mention_count = sum(text.count(kw) for kw in brand_keywords)
    brand_mention_rate  = brand_mention_count / max(n_words, 1)

    # Emotional / persuasive language
    positive_words = ['incredible','amazing','revolutionary','transform',
                      'success','growth','opportunity','best','leading',
                      'innovative','powerful','exciting','breakthrough',
                      'remarkable','outstanding']
    negative_words = ['risk','loss','decline','challenge','difficult',
                      'concern','problem','crisis','uncertainty','volatile']
    call_to_action = ['visit','learn more','find out','discover','join',
                      'contact','call','click','subscribe','follow']

    pos_count = sum(text.count(w) for w in positive_words)
    neg_count = sum(text.count(w) for w in negative_words)
    cta_count = sum(text.count(w) for w in call_to_action)

    # Uniqueness / vocabulary richness
    unique_words   = len(set(words))
    vocab_richness = unique_words / max(n_words, 1)

    # Numbers / statistics mentioned (adds credibility = memorability)
    number_count = len(re.findall(r'\b\d+[\.,]?\d*\b', text))
    number_rate  = number_count / max(n_words, 1)

    # Question count (rhetorical questions are memorable)
    question_count = text.count('?')

    # Empty transcript flag
    is_empty = int(n_words == 0)

    return [
        n_words, n_chars, n_sentences, avg_sent_len,
        brand_mention_count, brand_mention_rate,
        pos_count, neg_count, cta_count,
        vocab_richness, unique_words,
        number_count, number_rate,
        question_count, is_empty
    ]

text_feat_names = [
    'n_words','n_chars','n_sentences','avg_sent_len',
    'brand_mention_count','brand_mention_rate',
    'pos_count','neg_count','cta_count',
    'vocab_richness','unique_words',
    'number_count','number_rate',
    'question_count','is_empty'
]

X_text_hand = np.array([
    extract_text_features(row['id'], row['channelName'])
    for _, row in df.iterrows()
], dtype=float)

print(f"\nHand-crafted text features shape: {X_text_hand.shape}")

# Quick correlation check before modeling
print("\nCorrelation of hand-crafted features with targets:")
print(f"{'Feature':<22} {'vs Mem':>8} {'vs Brand':>8}")
print("-" * 42)
for i, fname in enumerate(text_feat_names):
    r_mem,   _ = spearmanr(X_text_hand[:, i], y_mem)
    r_brand, _ = spearmanr(X_text_hand[:, i], y_brand)
    print(f"{fname:<22} {r_mem:>8.4f} {r_brand:>8.4f}")


# ── 3. TF-IDF features ───────────────────────────────────────────────────────
corpus = [transcripts[vid_id] for vid_id in df['id']]

tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=2,           # must appear in at least 2 videos
    max_df=0.85,        # ignore terms in >85% of videos (too common)
    ngram_range=(1, 2), # unigrams + bigrams
    sublinear_tf=True,  # log(tf) scaling
    strip_accents='unicode',
    stop_words='english'
)
X_tfidf = tfidf.fit_transform(corpus)
print(f"\nTF-IDF matrix shape: {X_tfidf.shape}")

# Reduce with TruncatedSVD (LSA) — works directly on sparse matrix
svd = TruncatedSVD(n_components=100, random_state=42)
X_tfidf_lsa = svd.fit_transform(X_tfidf)
print(f"LSA-reduced shape:   {X_tfidf_lsa.shape}")
print(f"Variance explained:  {svd.explained_variance_ratio_.sum():.3f}")


# ── 4. Evaluate text features ────────────────────────────────────────────────
print("\n" + "="*60)
print("Text feature results (brand-isolated CV):")
print(f"{'Feature set':<30} {'Mem r':>8} {'Brand r':>8}")
print("-"*50)

# Hand-crafted only
r_m, r_b = cross_val_spearman(X_text_hand, y_mem, y_brand, df,
                               n_components=None, alpha=1.0)
print(f"{'Hand-crafted text':<30} {r_m:>8.4f} {r_b:>8.4f}")

# TF-IDF + LSA
r_m, r_b = cross_val_spearman(X_tfidf_lsa, y_mem, y_brand, df,
                               n_components=50, alpha=10.0)
print(f"{'TF-IDF + LSA(100)':<30} {r_m:>8.4f} {r_b:>8.4f}")

# Combined: hand-crafted + LSA
X_text_combined = np.concatenate([X_text_hand, X_tfidf_lsa], axis=1)
r_m, r_b = cross_val_spearman(X_text_combined, y_mem, y_brand, df,
                               n_components=80, alpha=10.0)
print(f"{'Hand-crafted + LSA':<30} {r_m:>8.4f} {r_b:>8.4f}")

# Text + best metadata
X_meta = np.column_stack([
    np.log1p(df['durationSeconds']),
    np.log1p(df['viewsCount']),
    np.log1p(df['likesCount']),
    np.log1p(df['engagementRate']),
    df['categoryId'].fillna(0),
    df['has_tags']
])
X_text_meta = np.concatenate([X_text_combined, X_meta], axis=1)
r_m, r_b = cross_val_spearman(X_text_meta, y_mem, y_brand, df,
                               n_components=80, alpha=10.0)
print(f"{'Text + Metadata':<30} {r_m:>8.4f} {r_b:>8.4f}")

# Text + R3D (best visual so far) + Metadata
X_full = np.concatenate([X_text_combined,
                          feature_matrices['R3D'],
                          X_meta], axis=1)
r_m, r_b = cross_val_spearman(X_full, y_mem, y_brand, df,
                               n_components=100, alpha=10.0)
print(f"{'Text + R3D + Metadata':<30} {r_m:>8.4f} {r_b:>8.4f}")


# ── 5. Cross-task feature: use mem score to predict brand ────────────────────
# Train memorability model, use its predictions as input to brand model
print("\n" + "="*60)
print("Cross-task: predicted memorability → brand memorability")

X_for_mem = X_text_meta
mem_preds  = np.zeros(len(df))

for fold_id in range(5):
    val_mask, train_mask = df['fold']==fold_id, df['fold']!=fold_id
    scaler = StandardScaler()
    X_tr  = scaler.fit_transform(X_for_mem[train_mask])
    X_val = scaler.transform(X_for_mem[val_mask])
    n_comp = min(80, X_tr.shape[1], X_tr.shape[0]-1)
    pca = PCA(n_components=n_comp)
    X_tr  = pca.fit_transform(X_tr)
    X_val = pca.transform(X_val)
    ridge = Ridge(alpha=10.0)
    ridge.fit(X_tr, y_mem[train_mask])
    mem_preds[val_mask] = ridge.predict(X_val)

# Now add predicted mem as feature for brand prediction
X_brand_augmented = np.column_stack([X_text_meta, mem_preds])
_, r_b = cross_val_spearman(X_brand_augmented, y_mem, y_brand, df,
                             n_components=80, alpha=10.0)
r_m = spearmanr(y_mem, mem_preds).correlation
print(f"  Memorability model Spearman r:        {r_m:.4f}")
print(f"  Brand model (with pred_mem) Spearman: {r_b:.4f}")

Missing STT files: 0
Empty transcripts: 3
Transcript word count — min:0, max:15613, mean:1529, median:574

Hand-crafted text features shape: (339, 15)

Correlation of hand-crafted features with targets:
Feature                  vs Mem vs Brand
------------------------------------------
n_words                 -0.0154   0.0074
n_chars                 -0.0199   0.0050
n_sentences             -0.0018   0.0067
avg_sent_len            -0.0473   0.0033
brand_mention_count     -0.0578   0.0600
brand_mention_rate      -0.0375   0.0726
pos_count                0.0458   0.0332
neg_count               -0.0524  -0.0368
cta_count               -0.0094   0.0252
vocab_richness          -0.0017  -0.0198
unique_words            -0.0274  -0.0019
number_count            -0.0418  -0.0224
number_rate             -0.1098  -0.0581
question_count           0.0189  -0.0351
is_empty                -0.0398   0.0285

TF-IDF matrix shape: (339, 5000)
LSA-reduced shape:   (339, 100)
Variance explained:  0.494

Text

## Step 5: Fix, Deepen, and Try Smarter Features

In [7]:
import numpy as np
import pandas as pd
import re
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ── 1. Fix NaN correlations ───────────────────────────────────────────────────
# NaN happens when a column is constant → spearmanr breaks
# Let's inspect and fix

print("Checking for constant/NaN columns in hand-crafted features:")
for i, fname in enumerate(text_feat_names):
    col = X_text_hand[:, i]
    n_unique = len(np.unique(col[~np.isnan(col)]))
    has_nan  = np.isnan(col).any()
    has_inf  = np.isinf(col).any()
    print(f"  {fname:<22} unique={n_unique:4d}  nan={has_nan}  inf={has_inf}")

# Fix: replace NaN/inf, then recompute correlations
X_text_fixed = np.nan_to_num(X_text_hand, nan=0.0, posinf=0.0, neginf=0.0)

print("\nFixed correlations:")
print(f"{'Feature':<22} {'vs Mem':>8} {'vs Brand':>8}")
print("-" * 42)
for i, fname in enumerate(text_feat_names):
    col = X_text_fixed[:, i]
    if len(np.unique(col)) < 2:
        print(f"{fname:<22} {'(constant)':>8}")
        continue
    r_mem,   _ = spearmanr(col, y_mem)
    r_brand, _ = spearmanr(col, y_brand)
    print(f"{fname:<22} {r_mem:>8.4f} {r_brand:>8.4f}")


# ── 2. Domain-specific text features for financial commercials ────────────────
def extract_financial_text_features(vid_id, channel_name):
    text = transcripts.get(vid_id, '').lower()
    words = text.split()
    n_words = max(len(words), 1)

    # --- Structural features ---
    n_sentences = max(len(re.split(r'[.!?]+', text)), 1)
    avg_sent_len = len(words) / n_sentences

    # --- Brand visibility ---
    # Clean channel name to key brand tokens
    stopwords = {'&','the','of','at','group','global','asset','management',
                 'investment','uk','us','emea','apac','solutions','managers',
                 'life','partners','general','fund','funds'}
    brand_tokens = [w for w in re.sub(r'[^a-z\s]','',channel_name.lower()).split()
                    if w not in stopwords and len(w) > 2]
    brand_mentions = sum(text.count(tok) for tok in brand_tokens)
    brand_rate     = brand_mentions / n_words

    # --- Financial domain vocabulary ---
    # These are words SPECIFIC to memorable financial content
    concrete_benefit = ['return','yield','dividend','profit','growth',
                        'gain','earn','income','save','wealth','fund',
                        'pension','retire','invest']
    risk_words       = ['risk','volatility','uncertainty','loss','protect',
                        'safe','secure','hedge','diversif']
    action_words     = ['discover','explore','learn','visit','contact',
                        'speak','start','begin','join','get']
    story_words      = ['story','journey','people','community','life',
                        'world','future','together','change','impact',
                        'help','support','believe']
    number_pattern   = re.findall(r'\b\d+\.?\d*\s*(?:%|percent|billion|million|trillion|bn|mn)?\b', text)

    concrete_count = sum(text.count(w) for w in concrete_benefit)
    risk_count     = sum(text.count(w) for w in risk_words)
    action_count   = sum(text.count(w) for w in action_words)
    story_count    = sum(text.count(w) for w in story_words)
    number_count   = len(number_pattern)

    # --- Lexical richness ---
    vocab_richness = len(set(words)) / n_words
    # Type-token ratio over first 100 words (normalizes for length)
    ttr_100 = len(set(words[:100])) / min(n_words, 100)

    # --- Readability proxy ---
    avg_word_len = np.mean([len(w) for w in words]) if words else 0

    # --- Repetition (memorable content often repeats key phrases) ---
    bigrams  = [' '.join(words[i:i+2]) for i in range(len(words)-1)]
    trigrams = [' '.join(words[i:i+3]) for i in range(len(words)-2)]
    bigram_repeat  = sum(1 for c in Counter(bigrams).values()  if c > 2)
    trigram_repeat = sum(1 for c in Counter(trigrams).values() if c > 1)

    # --- Speech density (words per second of video) ---
    duration = df.loc[df['id']==vid_id, 'durationSeconds'].values[0]
    speech_density = n_words / max(duration, 1)

    # --- Is the transcript very short relative to video length? ---
    # (Low density = mostly music/visuals = different memorability profile)
    low_speech = int(speech_density < 0.5)

    return [
        len(words), n_sentences, avg_sent_len,
        brand_mentions, brand_rate,
        concrete_count, concrete_count/n_words,
        risk_count, risk_count/n_words,
        action_count, action_count/n_words,
        story_count, story_count/n_words,
        number_count, number_count/n_words,
        vocab_richness, ttr_100, avg_word_len,
        bigram_repeat, trigram_repeat,
        speech_density, low_speech
    ]

feat_names_v2 = [
    'n_words','n_sentences','avg_sent_len',
    'brand_mentions','brand_rate',
    'concrete_count','concrete_rate',
    'risk_count','risk_rate',
    'action_count','action_rate',
    'story_count','story_rate',
    'number_count','number_rate',
    'vocab_richness','ttr_100','avg_word_len',
    'bigram_repeat','trigram_repeat',
    'speech_density','low_speech'
]

X_text_v2 = np.array([
    extract_financial_text_features(row['id'], row['channelName'])
    for _, row in df.iterrows()
], dtype=float)
X_text_v2 = np.nan_to_num(X_text_v2, nan=0.0, posinf=0.0, neginf=0.0)

print("\n" + "="*55)
print("Domain-specific feature correlations with targets:")
print(f"{'Feature':<20} {'vs Mem':>8} {'vs Brand':>8}")
print("-" * 40)
for i, fname in enumerate(feat_names_v2):
    if len(np.unique(X_text_v2[:, i])) < 2:
        continue
    r_m, _ = spearmanr(X_text_v2[:, i], y_mem)
    r_b, _ = spearmanr(X_text_v2[:, i], y_brand)
    marker = ' ◄' if abs(r_m) > 0.1 or abs(r_b) > 0.1 else ''
    print(f"{fname:<20} {r_m:>8.4f} {r_b:>8.4f}{marker}")


# ── 3. Better TF-IDF: separate short vs long videos ──────────────────────────
# Short videos (ads) vs long videos (talks/webinars) have totally different
# language patterns — mixing them in one TF-IDF hurts

duration_median = df['durationSeconds'].median()
is_short = df['durationSeconds'] <= duration_median

print(f"\nShort videos (≤{duration_median:.0f}s): {is_short.sum()}")
print(f"Long videos  (>{duration_median:.0f}s): {(~is_short).sum()}")

# TF-IDF on full corpus but with better params
tfidf_v2 = TfidfVectorizer(
    max_features=3000,
    min_df=3,
    max_df=0.80,
    ngram_range=(1, 3),   # include trigrams
    sublinear_tf=True,
    strip_accents='unicode',
    stop_words='english',
    analyzer='word'
)
X_tfidf_v2   = tfidf_v2.fit_transform(corpus)
svd_v2       = TruncatedSVD(n_components=80, random_state=42)
X_lsa_v2     = svd_v2.fit_transform(X_tfidf_v2)


# ── 4. Full evaluation: all combinations ─────────────────────────────────────
X_meta = np.nan_to_num(np.column_stack([
    np.log1p(df['durationSeconds']),
    np.log1p(df['viewsCount']),
    np.log1p(df['likesCount']),
    np.log1p(df['engagementRate']),
    df['categoryId'].fillna(0),
    df['has_tags'].fillna(0),
    is_short.astype(float)          # short/long flag
]), nan=0.0)

print("\n" + "="*60)
print(f"{'Feature combination':<38} {'Mem r':>7} {'Brand r':>8}")
print("-"*56)

combos = {
    'Metadata only':          X_meta,
    'Domain text features':   X_text_v2,
    'LSA v2 (80d)':           X_lsa_v2,
    'Domain text + metadata': np.concatenate([X_text_v2, X_meta], axis=1),
    'LSA + metadata':         np.concatenate([X_lsa_v2,  X_meta], axis=1),
    'Domain + LSA':           np.concatenate([X_text_v2, X_lsa_v2], axis=1),
    'Domain + LSA + meta':    np.concatenate([X_text_v2, X_lsa_v2, X_meta], axis=1),
    'Above + R3D':            np.concatenate([X_text_v2, X_lsa_v2, X_meta,
                                              feature_matrices['R3D']], axis=1),
    'Above + ViT':            np.concatenate([X_text_v2, X_lsa_v2, X_meta,
                                              feature_matrices['ViT']], axis=1),
    'ALL modalities':         np.concatenate([X_text_v2, X_lsa_v2, X_meta] +
                                             list(feature_matrices.values()), axis=1),
}

best_mem, best_brand, best_combo = -99, -99, ''
for name, X in combos.items():
    n_comp = min(100, X.shape[1], 260)
    r_m, r_b = cross_val_spearman(X, y_mem, y_brand, df,
                                   n_components=n_comp, alpha=10.0)
    marker = ' ◄ BEST' if r_m > best_mem else ''
    if r_m > best_mem:
        best_mem, best_brand, best_combo = r_m, r_b, name
    print(f"{name:<38} {r_m:>7.4f} {r_b:>8.4f}{marker}")

print(f"\nBest combo: '{best_combo}' → Mem={best_mem:.4f}, Brand={best_brand:.4f}")

Checking for constant/NaN columns in hand-crafted features:
  n_words                unique= 302  nan=False  inf=False
  n_chars                unique= 322  nan=False  inf=False
  n_sentences            unique= 150  nan=False  inf=False
  avg_sent_len           unique= 320  nan=False  inf=False
  brand_mention_count    unique=  27  nan=False  inf=False
  brand_mention_rate     unique= 163  nan=False  inf=False
  pos_count              unique=  30  nan=False  inf=False
  neg_count              unique=  32  nan=False  inf=False
  cta_count              unique=  33  nan=False  inf=False
  vocab_richness         unique= 325  nan=False  inf=False
  unique_words           unique= 280  nan=False  inf=False
  number_count           unique=  52  nan=False  inf=False
  number_rate            unique= 235  nan=False  inf=False
  question_count         unique=  38  nan=False  inf=False
  is_empty               unique=   2  nan=False  inf=False

Fixed correlations:
Feature                  vs Mem vs

In [8]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
import warnings
warnings.filterwarnings('ignore')

# ── 1. Build the best feature set from what we know works ────────────────────
# Only use features that showed positive/meaningful signal

# Metadata (best performer so far)
X_meta = np.nan_to_num(np.column_stack([
    np.log1p(df['durationSeconds']),
    np.log1p(df['viewsCount']),
    np.log1p(df['likesCount']),
    np.log1p(df['engagementRate']),
    df['categoryId'].fillna(0),
    df['has_tags'].fillna(0),
    (df['durationSeconds'] <= df['durationSeconds'].median()).astype(float),
    np.log1p(df['commentsCount']),
    np.log1p(df['dislikesCount']),
    np.log1p(df['favouritesCount']),
]), nan=0.0)

# Selective text features — only keep ones with |r| > 0.05 with either target
# From our analysis: brand_rate, action_rate, speech_density, low_speech,
#                    concrete_rate (negative signal), risk_rate (negative signal)
X_text_selective = X_text_v2[:, [
    feat_names_v2.index('brand_mentions'),
    feat_names_v2.index('brand_rate'),
    feat_names_v2.index('action_count'),
    feat_names_v2.index('action_rate'),
    feat_names_v2.index('concrete_rate'),
    feat_names_v2.index('risk_rate'),
    feat_names_v2.index('number_rate'),
    feat_names_v2.index('speech_density'),
    feat_names_v2.index('low_speech'),
    feat_names_v2.index('avg_sent_len'),
    feat_names_v2.index('n_words'),
    feat_names_v2.index('vocab_richness'),
]]
X_text_selective = np.nan_to_num(X_text_selective, nan=0.0, posinf=0.0, neginf=0.0)

# LSA (shows small positive signal alone)
# Use a smaller LSA — less overfitting
svd_small = TruncatedSVD(n_components=30, random_state=42)
X_lsa_small = svd_small.fit_transform(X_tfidf_v2)

print(f"Metadata features:        {X_meta.shape[1]}")
print(f"Selective text features:  {X_text_selective.shape[1]}")
print(f"LSA features:             {X_lsa_small.shape[1]}")


# ── 2. Annotation-count weighting ────────────────────────────────────────────
# Videos with more annotations have more reliable scores
# Use nb_annotations as sample weights

nb_ann = df['nb_annotations'].values.astype(float)
weights = nb_ann / nb_ann.mean()  # normalize around 1.0
print(f"\nAnnotation counts — min:{nb_ann.min():.0f}, "
      f"max:{nb_ann.max():.0f}, mean:{nb_ann.mean():.1f}")


# ── 3. Weighted cross-validation evaluator ───────────────────────────────────
def cross_val_weighted(X, y_mem, y_brand, df, weights,
                        model_type='ridge', alpha=10.0,
                        n_components=None, use_weights=True):
    preds_mem   = np.zeros(len(df))
    preds_brand = np.zeros(len(df))

    for fold_id in range(5):
        val_mask   = (df['fold'] == fold_id).values
        train_mask = ~val_mask

        X_tr, X_val = X[train_mask], X[val_mask]
        w_tr = weights[train_mask] if use_weights else None

        scaler = RobustScaler()  # more robust to outliers than StandardScaler
        X_tr  = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)

        if n_components is not None:
            n_comp = min(n_components, X_tr.shape[1], X_tr.shape[0]-1)
            pca = PCA(n_components=n_comp)
            X_tr  = pca.fit_transform(X_tr)
            X_val = pca.transform(X_val)

        for y, preds in [(y_mem, preds_mem), (y_brand, preds_brand)]:
            if model_type == 'ridge':
                m = Ridge(alpha=alpha)
                m.fit(X_tr, y[train_mask], sample_weight=w_tr)
            elif model_type == 'svr':
                m = SVR(kernel='rbf', C=alpha, epsilon=0.05)
                m.fit(X_tr, y[train_mask],
                      sample_weight=w_tr if use_weights else None)
            elif model_type == 'gbm':
                m = GradientBoostingRegressor(
                    n_estimators=100, max_depth=3,
                    learning_rate=0.05, subsample=0.8, random_state=42)
                m.fit(X_tr, y[train_mask], sample_weight=w_tr)
            elif model_type == 'elasticnet':
                m = ElasticNet(alpha=alpha, l1_ratio=0.5, max_iter=5000)
                m.fit(X_tr, y[train_mask], sample_weight=w_tr)
            preds[val_mask] = m.predict(X_val)

    return (spearmanr(y_mem,   preds_mem).correlation,
            spearmanr(y_brand, preds_brand).correlation)


# ── 4. Model comparison on best feature set ───────────────────────────────────
X_best = np.concatenate([X_meta, X_text_selective], axis=1)
X_best_lsa = np.concatenate([X_meta, X_text_selective, X_lsa_small], axis=1)

print("\n" + "="*65)
print(f"{'Config':<42} {'Mem r':>7} {'Brand r':>8}")
print("-"*60)

configs = [
    ('Meta + SelectText | Ridge | no weight',
     X_best,     'ridge', 10.0,  None,  False),
    ('Meta + SelectText | Ridge | weighted',
     X_best,     'ridge', 10.0,  None,  True),
    ('Meta + SelectText | Ridge a=1 | weighted',
     X_best,     'ridge', 1.0,   None,  True),
    ('Meta + SelectText | ElasticNet | weighted',
     X_best,     'elasticnet', 0.01, None, True),
    ('Meta + SelectText | SVR | weighted',
     X_best,     'svr',   1.0,   None,  True),
    ('Meta + SelectText | GBM | weighted',
     X_best,     'gbm',   None,  None,  True),
    ('Meta+SelectText+LSA | Ridge | weighted',
     X_best_lsa, 'ridge', 10.0,  50,    True),
    ('Meta+SelectText+LSA | GBM | weighted',
     X_best_lsa, 'gbm',   None,  None,  True),
]

best_mem = -99
for name, X, model, alpha, n_comp, use_w in configs:
    r_m, r_b = cross_val_weighted(X, y_mem, y_brand, df, weights,
                                   model_type=model, alpha=alpha if alpha else 10.0,
                                   n_components=n_comp, use_weights=use_w)
    marker = ' ◄' if r_m > best_mem else ''
    if r_m > best_mem: best_mem = r_m
    print(f"{name:<42} {r_m:>7.4f} {r_b:>8.4f}{marker}")


# ── 5. Feature selection: let the data choose ────────────────────────────────
print("\n" + "="*65)
print("Feature selection (SelectKBest on Meta+SelectText):")
print(f"{'k features':<15} {'Mem r':>7} {'Brand r':>8}")
print("-"*35)

for k in [3, 5, 8, 10, 12, 'all']:
    preds_mem   = np.zeros(len(df))
    preds_brand = np.zeros(len(df))
    for fold_id in range(5):
        val_mask   = (df['fold'] == fold_id).values
        train_mask = ~val_mask
        X_tr, X_val = X_best[train_mask], X_best[val_mask]
        w_tr = weights[train_mask]
        scaler = RobustScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)
        kk = X_tr.shape[1] if k == 'all' else k
        for y, preds in [(y_mem, preds_mem), (y_brand, preds_brand)]:
            sel = SelectKBest(f_regression, k=kk)
            X_tr_sel  = sel.fit_transform(X_tr, y[train_mask])
            X_val_sel = sel.transform(X_val)
            m = Ridge(alpha=10.0)
            m.fit(X_tr_sel, y[train_mask], sample_weight=w_tr)
            preds[val_mask] = m.predict(X_val_sel)
    r_m = spearmanr(y_mem,   preds_mem).correlation
    r_b = spearmanr(y_brand, preds_brand).correlation
    print(f"{str(k):<15} {r_m:>7.4f} {r_b:>8.4f}")


# ── 6. Per-target separate feature selection ─────────────────────────────────
print("\n" + "="*65)
print("Best individual features per target (Spearman):")
X_all_candidates = np.concatenate([X_meta, X_text_selective, X_lsa_small], axis=1)
all_names = (
    [f'meta_{i}' for i in range(X_meta.shape[1])] +
    [feat_names_v2[i] for i in [
        feat_names_v2.index('brand_mentions'),
        feat_names_v2.index('brand_rate'),
        feat_names_v2.index('action_count'),
        feat_names_v2.index('action_rate'),
        feat_names_v2.index('concrete_rate'),
        feat_names_v2.index('risk_rate'),
        feat_names_v2.index('number_rate'),
        feat_names_v2.index('speech_density'),
        feat_names_v2.index('low_speech'),
        feat_names_v2.index('avg_sent_len'),
        feat_names_v2.index('n_words'),
        feat_names_v2.index('vocab_richness'),
    ]] +
    [f'lsa_{i}' for i in range(X_lsa_small.shape[1])]
)

r_mem_list   = [(spearmanr(X_all_candidates[:,i], y_mem).correlation,   n)
                for i, n in enumerate(all_names)]
r_brand_list = [(spearmanr(X_all_candidates[:,i], y_brand).correlation, n)
                for i, n in enumerate(all_names)]

print("\nTop 10 for Memorability:")
for r, n in sorted(r_mem_list, key=lambda x: abs(x[0]), reverse=True)[:10]:
    print(f"  {n:<25} {r:>7.4f}")

print("\nTop 10 for Brand Memorability:")
for r, n in sorted(r_brand_list, key=lambda x: abs(x[0]), reverse=True)[:10]:
    print(f"  {n:<25} {r:>7.4f}")

Metadata features:        10
Selective text features:  12
LSA features:             30

Annotation counts — min:12, max:51, mean:17.7

Config                                       Mem r  Brand r
------------------------------------------------------------
Meta + SelectText | Ridge | no weight      -0.0342   0.0444 ◄
Meta + SelectText | Ridge | weighted       -0.0494   0.0363
Meta + SelectText | Ridge a=1 | weighted   -0.0203   0.0488 ◄
Meta + SelectText | ElasticNet | weighted  -0.0702  -0.0919
Meta + SelectText | SVR | weighted         -0.0043  -0.0063 ◄
Meta + SelectText | GBM | weighted         -0.2143  -0.0739
Meta+SelectText+LSA | Ridge | weighted     -0.0299  -0.0271
Meta+SelectText+LSA | GBM | weighted       -0.1039  -0.0378

Feature selection (SelectKBest on Meta+SelectText):
k features        Mem r  Brand r
-----------------------------------
3               -0.0629  -0.1158
5               -0.0251  -0.1024
8               -0.0214  -0.0721
10              -0.0269  -0.0944
12  

In [9]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, rankdata
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# ── 1. Audit all metadata columns ────────────────────────────────────────────
meta_col_names = [
    'log_duration', 'log_views', 'log_likes', 'log_engagement',
    'categoryId', 'has_tags', 'is_short',
    'log_comments', 'log_dislikes', 'log_favourites'
]

print("Full audit of all candidate features vs both targets:")
print(f"{'Feature':<25} {'vs Mem':>8} {'vs Brand':>8} {'note'}")
print("-" * 65)

all_features = {}

for i, name in enumerate(meta_col_names):
    col = X_meta[:, i]
    n_unique = len(np.unique(col[~np.isnan(col)]))
    if n_unique < 2:
        print(f"{name:<25} {'CONSTANT — SKIP':>20}")
        continue
    r_m, _ = spearmanr(col, y_mem)
    r_b, _ = spearmanr(col, y_brand)
    note = ' ◄' if abs(r_m) > 0.10 or abs(r_b) > 0.10 else ''
    print(f"{name:<25} {r_m:>8.4f} {r_b:>8.4f}{note}")
    all_features[name] = col

# Text features
for i, name in enumerate(feat_names_v2):
    col = X_text_v2[:, i]
    n_unique = len(np.unique(col[~np.isnan(col)]))
    if n_unique < 2:
        continue
    col = np.nan_to_num(col)
    r_m, _ = spearmanr(col, y_mem)
    r_b, _ = spearmanr(col, y_brand)
    note = ' ◄' if abs(r_m) > 0.10 or abs(r_b) > 0.10 else ''
    if abs(r_m) > 0.05 or abs(r_b) > 0.07:
        print(f"{name:<25} {r_m:>8.4f} {r_b:>8.4f}{note}")
    all_features[name] = col

# LSA dims
for i in range(X_lsa_small.shape[1]):
    col = X_lsa_small[:, i]
    r_m, _ = spearmanr(col, y_mem)
    r_b, _ = spearmanr(col, y_brand)
    if abs(r_m) > 0.12 or abs(r_b) > 0.10:
        name = f'lsa_{i}'
        note = ' ◄'
        print(f"{name:<25} {r_m:>8.4f} {r_b:>8.4f}{note}")
        all_features[name] = col


# ── 2. Rank aggregation — NO model fitting ────────────────────────────────────
# Instead of fitting weights, we use each feature's known correlation direction
# to flip signs, then average ranks. Zero overfitting risk.

def rank_aggregate_cv(feature_dict, y_mem, y_brand, df):
    """
    For each fold:
      - Compute Spearman r of each feature vs target ON TRAIN SET
      - Sign-flip features that are negatively correlated
      - Average the ranks across features
      - Evaluate on val set
    No model fitting — just rank averaging.
    """
    preds_mem   = np.zeros(len(df))
    preds_brand = np.zeros(len(df))
    feat_names  = list(feature_dict.keys())
    X_all       = np.column_stack(list(feature_dict.values()))

    for fold_id in range(5):
        val_mask   = (df['fold'] == fold_id).values
        train_mask = ~val_mask

        X_tr  = X_all[train_mask]
        X_val = X_all[val_mask]
        n_val = val_mask.sum()

        # Compute direction on train set for each target
        for y, preds in [(y_mem, preds_mem), (y_brand, preds_brand)]:
            y_tr = y[train_mask]
            directions = []
            valid_cols  = []
            for j in range(X_tr.shape[1]):
                col = X_tr[:, j]
                if len(np.unique(col)) < 2:
                    continue
                r, _ = spearmanr(col, y_tr)
                if abs(r) < 0.03:   # skip near-zero features this fold
                    continue
                directions.append(np.sign(r))
                valid_cols.append(j)

            if not valid_cols:
                preds[val_mask] = y_tr.mean()
                continue

            # Rank each feature on val set, flip direction, average
            agg = np.zeros(n_val)
            for j, sign in zip(valid_cols, directions):
                col_val = X_val[:, j]
                ranked  = rankdata(col_val).astype(float)
                agg    += sign * ranked

            preds[val_mask] = agg  # higher = more memorable

    r_m = spearmanr(y_mem,   preds_mem).correlation
    r_b = spearmanr(y_brand, preds_brand).correlation
    return r_m, r_b, preds_mem, preds_brand


# ── 3. Try rank aggregation with different feature subsets ───────────────────
print("\n" + "="*60)
print("Rank aggregation results (no model fitting):")
print(f"{'Feature subset':<35} {'Mem r':>7} {'Brand r':>8}")
print("-"*55)

# All features
r_m, r_b, _, _ = rank_aggregate_cv(all_features, y_mem, y_brand, df)
print(f"{'All features':<35} {r_m:>7.4f} {r_b:>8.4f}")

# Only metadata
meta_feats = {k: v for k, v in all_features.items()
              if k.startswith('meta_') or k in
              ['log_duration','log_views','log_likes',
               'log_engagement','categoryId','is_short',
               'log_comments','log_dislikes']}
r_m, r_b, _, _ = rank_aggregate_cv(meta_feats, y_mem, y_brand, df)
print(f"{'Metadata only':<35} {r_m:>7.4f} {r_b:>8.4f}")

# Only text
text_feats = {k: v for k, v in all_features.items()
              if k in feat_names_v2}
r_m, r_b, _, _ = rank_aggregate_cv(text_feats, y_mem, y_brand, df)
print(f"{'Text features only':<35} {r_m:>7.4f} {r_b:>8.4f}")

# Only LSA
lsa_feats = {k: v for k, v in all_features.items()
             if k.startswith('lsa_')}
r_m, r_b, _, _ = rank_aggregate_cv(lsa_feats, y_mem, y_brand, df)
print(f"{'LSA features only':<35} {r_m:>7.4f} {r_b:>8.4f}")

# Metadata + text (no LSA)
meta_text = {**meta_feats, **text_feats}
r_m, r_b, _, _ = rank_aggregate_cv(meta_text, y_mem, y_brand, df)
print(f"{'Meta + text':<35} {r_m:>7.4f} {r_b:>8.4f}")


# ── 4. Single feature Ridge — one predictor at a time ────────────────────────
# Eliminates collinearity entirely
print("\n" + "="*60)
print("Single-feature Ridge CV (eliminates collinearity):")
print(f"{'Feature':<25} {'Mem r':>7} {'Brand r':>8}")
print("-"*45)

single_best_mem, single_best_brand = {}, {}
for name, col in all_features.items():
    if np.isnan(col).any() or len(np.unique(col)) < 2:
        continue
    X_single = col.reshape(-1, 1)
    r_m, r_b = cross_val_spearman(X_single, y_mem, y_brand, df,
                                   n_components=None, alpha=1.0)
    if abs(r_m) > 0.05 or abs(r_b) > 0.07:
        print(f"{name:<25} {r_m:>7.4f} {r_b:>8.4f}")
    single_best_mem[name]   = r_m
    single_best_brand[name] = r_b

top_mem   = sorted(single_best_mem,   key=lambda k: single_best_mem[k],   reverse=True)[:5]
top_brand = sorted(single_best_brand, key=lambda k: single_best_brand[k], reverse=True)[:5]

print(f"\nTop 5 single features for memorability:     {top_mem}")
print(f"Top 5 single features for brand memorability: {top_brand}")


# ── 5. Ensemble of top single-feature models ─────────────────────────────────
print("\n" + "="*60)
print("Ensemble of top single-feature Ridge models:")
print(f"{'Top-k ensemble':<20} {'Mem r':>7} {'Brand r':>8}")
print("-"*40)

for top_k in [2, 3, 5, 8]:
    # For memorability
    top_k_mem = sorted(single_best_mem,
                       key=lambda k: single_best_mem[k], reverse=True)[:top_k]
    top_k_brand = sorted(single_best_brand,
                         key=lambda k: single_best_brand[k], reverse=True)[:top_k]

    preds_mem   = np.zeros(len(df))
    preds_brand = np.zeros(len(df))

    for fold_id in range(5):
        val_mask   = (df['fold'] == fold_id).values
        train_mask = ~val_mask

        # Memorability ensemble
        fold_preds_m = []
        for name in top_k_mem:
            col = all_features[name].reshape(-1, 1)
            scaler = RobustScaler()
            X_tr  = scaler.fit_transform(col[train_mask])
            X_val = scaler.transform(col[val_mask])
            m = Ridge(alpha=1.0)
            m.fit(X_tr, y_mem[train_mask])
            fold_preds_m.append(m.predict(X_val))
        preds_mem[val_mask] = np.mean(fold_preds_m, axis=0)

        # Brand ensemble
        fold_preds_b = []
        for name in top_k_brand:
            col = all_features[name].reshape(-1, 1)
            scaler = RobustScaler()
            X_tr  = scaler.fit_transform(col[train_mask])
            X_val = scaler.transform(col[val_mask])
            m = Ridge(alpha=1.0)
            m.fit(X_tr, y_brand[train_mask])
            fold_preds_b.append(m.predict(X_val))
        preds_brand[val_mask] = np.mean(fold_preds_b, axis=0)

    r_m = spearmanr(y_mem,   preds_mem).correlation
    r_b = spearmanr(y_brand, preds_brand).correlation
    print(f"Top-{top_k:<17} {r_m:>7.4f} {r_b:>8.4f}")

Full audit of all candidate features vs both targets:
Feature                     vs Mem vs Brand note
-----------------------------------------------------------------
log_duration               -0.0228   0.0091
log_views                   0.1362   0.0799 ◄
log_likes                  -0.1202  -0.0974 ◄
log_engagement             -0.1300  -0.1141 ◄
categoryId                 -0.0344  -0.0415
has_tags                   -0.1591  -0.1198 ◄
is_short                    0.0318   0.0463
log_comments               -0.0274   0.0001
log_dislikes               -0.0641  -0.0540
log_favourites                 CONSTANT — SKIP
brand_mentions             -0.0581   0.0800
brand_rate                 -0.0392   0.0943
concrete_count             -0.1010  -0.0512 ◄
concrete_rate              -0.1077  -0.0475 ◄
risk_count                 -0.1035  -0.0357 ◄
risk_rate                  -0.0938  -0.0289
action_rate                 0.0664   0.0826
story_rate                  0.0248  -0.0865
number_rate           

In [10]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

# ── 1. What does lsa_28 actually represent? ───────────────────────────────────
# Inspect the top words loading onto the most predictive LSA dimensions

print("="*60)
print("WHAT DO THE KEY LSA DIMENSIONS REPRESENT?")
print("="*60)

feature_names = tfidf_v2.get_feature_names_out()
components    = svd_v2.components_  # shape: (n_components, n_vocab)

for dim_idx, dim_name in [(28, 'lsa_28'), (1, 'lsa_1'),
                           (6, 'lsa_6'), (19, 'lsa_19')]:
    comp = components[dim_idx]
    top_pos = np.argsort(comp)[-15:][::-1]
    top_neg = np.argsort(comp)[:15]

    r_m, _ = spearmanr(X_lsa_small[:, dim_idx], y_mem)
    r_b, _ = spearmanr(X_lsa_small[:, dim_idx], y_brand)

    print(f"\n{dim_name}  (r_mem={r_m:.3f}, r_brand={r_b:.3f})")
    print(f"  HIGH score words: {', '.join(feature_names[top_pos])}")
    print(f"  LOW score words:  {', '.join(feature_names[top_neg])}")

    # Show sample videos at extremes
    scores  = X_lsa_small[:, dim_idx]
    top_vids = np.argsort(scores)[-3:][::-1]
    bot_vids = np.argsort(scores)[:3]
    print("  HIGH score videos:")
    for i in top_vids:
        row = df.iloc[i]
        print(f"    [{scores[i]:.2f}] {row['channelName']}: {row['title'][:60]}")
    print("  LOW score videos:")
    for i in bot_vids:
        row = df.iloc[i]
        print(f"    [{scores[i]:.2f}] {row['channelName']}: {row['title'][:60]}")


# ── 2. Optimize LSA rank aggregation — our best method ───────────────────────
print("\n" + "="*60)
print("OPTIMIZING LSA RANK AGGREGATION")
print(f"{'Config':<45} {'Mem r':>7} {'Brand r':>8}")
print("-"*62)

# Try different numbers of LSA components
for n_comp in [20, 30, 50, 80, 100, 150]:
    svd_test = TruncatedSVD(n_components=n_comp, random_state=42)
    X_lsa_test = svd_test.fit_transform(X_tfidf_v2)
    lsa_feats_test = {f'lsa_{i}': X_lsa_test[:, i]
                      for i in range(n_comp)}
    r_m, r_b, _, _ = rank_aggregate_cv(lsa_feats_test, y_mem, y_brand, df)
    print(f"{'LSA rank-agg n_comp='+str(n_comp):<45} {r_m:>7.4f} {r_b:>8.4f}")

# Try different TF-IDF vocabularies
for max_feat, ngram in [(2000,(1,2)), (5000,(1,2)),
                         (3000,(1,3)), (3000,(2,3))]:
    tfidf_t = TfidfVectorizer(max_features=max_feat, min_df=2,
                               max_df=0.85, ngram_range=ngram,
                               sublinear_tf=True, stop_words='english')
    X_tfidf_t = tfidf_t.fit_transform(corpus)
    svd_t = TruncatedSVD(n_components=50, random_state=42)
    X_lsa_t = svd_t.fit_transform(X_tfidf_t)
    lsa_feats_t = {f'lsa_{i}': X_lsa_t[:, i] for i in range(50)}
    r_m, r_b, _, _ = rank_aggregate_cv(lsa_feats_t, y_mem, y_brand, df)
    label = f'LSA(50) tfidf max={max_feat} ngram={ngram}'
    print(f"{label:<45} {r_m:>7.4f} {r_b:>8.4f}")


# ── 3. Brand debiasing — remove Goldman Sachs distribution shift ─────────────
# The core problem: Goldman Sachs (fold 0) has very different feature
# distributions. Solution: subtract per-channel mean before modeling.

print("\n" + "="*60)
print("BRAND DEBIASING (subtract per-channel feature mean)")

def rank_aggregate_debiased(feature_dict, y_mem, y_brand, df):
    """Same as rank_aggregate_cv but subtracts per-channel mean per fold."""
    preds_mem   = np.zeros(len(df))
    preds_brand = np.zeros(len(df))
    feat_names  = list(feature_dict.keys())
    X_all       = np.column_stack(list(feature_dict.values()))

    for fold_id in range(5):
        val_mask   = (df['fold'] == fold_id).values
        train_mask = ~val_mask

        X_tr_raw  = X_all[train_mask].copy()
        X_val_raw = X_all[val_mask].copy()

        # Subtract per-channel mean computed on train set
        channels_train = df['channelName'].values[train_mask]
        channels_val   = df['channelName'].values[val_mask]
        global_mean    = X_tr_raw.mean(axis=0)

        for ch in np.unique(channels_train):
            mask_ch = channels_train == ch
            if mask_ch.sum() > 1:
                ch_mean = X_tr_raw[mask_ch].mean(axis=0)
                X_tr_raw[mask_ch] -= (ch_mean - global_mean)

        # For val channels: use train channel mean if available, else global
        for ch in np.unique(channels_val):
            mask_ch_val = channels_val == ch
            mask_ch_tr  = channels_train == ch
            if mask_ch_tr.sum() > 1:
                ch_mean = X_all[train_mask][mask_ch_tr].mean(axis=0)
            else:
                ch_mean = global_mean
            X_val_raw[mask_ch_val] -= (ch_mean - global_mean)

        for y, preds in [(y_mem, preds_mem), (y_brand, preds_brand)]:
            y_tr = y[train_mask]
            directions, valid_cols = [], []
            for j in range(X_tr_raw.shape[1]):
                col = X_tr_raw[:, j]
                if len(np.unique(col)) < 2: continue
                r, _ = spearmanr(col, y_tr)
                if abs(r) < 0.03: continue
                directions.append(np.sign(r))
                valid_cols.append(j)

            if not valid_cols:
                preds[val_mask] = y_tr.mean()
                continue

            from scipy.stats import rankdata
            agg = np.zeros(val_mask.sum())
            for j, sign in zip(valid_cols, directions):
                agg += sign * rankdata(X_val_raw[:, j]).astype(float)
            preds[val_mask] = agg

    return (spearmanr(y_mem,   preds_mem).correlation,
            spearmanr(y_brand, preds_brand).correlation,
            preds_mem, preds_brand)

# Best LSA config from above
svd_best = TruncatedSVD(n_components=50, random_state=42)
X_lsa_best = svd_best.fit_transform(X_tfidf_v2)
lsa_best_feats = {f'lsa_{i}': X_lsa_best[:, i] for i in range(50)}

r_m, r_b, pm, pb = rank_aggregate_debiased(
    lsa_best_feats, y_mem, y_brand, df)
print(f"  LSA(50) debiased rank-agg:  Mem={r_m:.4f}  Brand={r_b:.4f}")

# Without debiasing for comparison
r_m2, r_b2, _, _ = rank_aggregate_cv(lsa_best_feats, y_mem, y_brand, df)
print(f"  LSA(50) standard rank-agg:  Mem={r_m2:.4f}  Brand={r_b2:.4f}")


# ── 4. Per-fold breakdown of best method ─────────────────────────────────────
print("\n" + "="*60)
print("PER-FOLD BREAKDOWN — LSA rank aggregation (best config)")

X_all_lsa = np.column_stack([X_lsa_best[:, i] for i in range(50)])
from scipy.stats import rankdata

for fold_id in range(5):
    val_mask   = (df['fold'] == fold_id).values
    train_mask = ~val_mask

    # Quick rank agg for this fold
    agg_mem = np.zeros(val_mask.sum())
    agg_brd = np.zeros(val_mask.sum())
    count = 0
    for j in range(50):
        col_tr  = X_all_lsa[train_mask, j]
        col_val = X_all_lsa[val_mask,   j]
        r_m_tr, _ = spearmanr(col_tr, y_mem[train_mask])
        r_b_tr, _ = spearmanr(col_tr, y_brand[train_mask])
        if abs(r_m_tr) > 0.03:
            agg_mem += np.sign(r_m_tr) * rankdata(col_val).astype(float)
            count += 1
        if abs(r_b_tr) > 0.03:
            agg_brd += np.sign(r_b_tr) * rankdata(col_val).astype(float)

    r_m = spearmanr(y_mem[val_mask],   agg_mem).correlation
    r_b = spearmanr(y_brand[val_mask], agg_brd).correlation
    channels = df[val_mask]['channelName'].unique()[:2].tolist()
    print(f"  Fold {fold_id} (n={val_mask.sum():3d}) "
          f"Mem={r_m:.4f} Brand={r_b:.4f}  {channels}...")


# ── 5. Final summary ─────────────────────────────────────────────────────────
print("\n" + "="*60)
print("BEST RESULTS SUMMARY SO FAR")
print(f"  LSA rank aggregation (no fitting): Mem≈0.17, Brand≈0.14")
print(f"  Top-2 ensemble Ridge:              Mem≈0.07, Brand≈0.10")
print(f"  Metadata only (Ridge):             Mem≈0.06, Brand≈0.06")
print(f"\nConclusion: LSA topic features + rank aggregation")
print(f"is our most robust method. Next step: build final")
print(f"submission pipeline using this as the core.")

WHAT DO THE KEY LSA DIMENSIONS REPRESENT?

lsa_28  (r_mem=-0.167, r_brand=-0.183)
  HIGH score words: work, career, cover, clients, home, ve, yeah, payments, time, select, bonds, pandemic, choose, office, payment
  LOW score words:  etfs, women, building, uk, ask, going, want, right thing, bank, wind, life, people, senior, week, water
  HIGH score videos:
    [0.29] Invesco: Agents of Innovation: Kim
    [0.25] HSBC UK: “I need a big deposit to buy a home” | Mortgage myth busters
    [0.25] Invesco: Agents of Innovation: Queso
  LOW score videos:
    [-0.20] Vanguard: Qualifying for Social Security
    [-0.20] Vanguard: When to Take Social Security
    [-0.19] Invesco: Invesco QQQ - What are the advantages of an ETF

lsa_1  (r_mem=-0.146, r_brand=-0.035)
  HIGH score words: people, work, music, working, really, know, life, just, things, make, experience, want, career, women, like
  LOW score words:  market, markets, growth, equities, rates, economy, investors, inflation, equity, stocks

## Final Submission Pipeline

In [11]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, rankdata
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── 1. Find optimal n_comp per target separately ──────────────────────────────
print("="*60)
print("FINDING OPTIMAL n_comp PER TARGET")
print(f"{'n_comp':<10} {'Mem r':>8} {'Brand r':>8}")
print("-"*30)

best_mem_score,   best_mem_ncomp   = -99, 50
best_brand_score, best_brand_ncomp = -99, 50

for n_comp in [50, 80, 100, 120, 150, 200]:
    svd_t = TruncatedSVD(n_components=n_comp, random_state=42)
    X_lsa_t = svd_t.fit_transform(X_tfidf_v2)
    feats_t = {f'lsa_{i}': X_lsa_t[:, i] for i in range(n_comp)}
    r_m, r_b, _, _ = rank_aggregate_cv(feats_t, y_mem, y_brand, df)
    print(f"{n_comp:<10} {r_m:>8.4f} {r_b:>8.4f}")
    if r_m > best_mem_score:
        best_mem_score, best_mem_ncomp = r_m, n_comp
    if r_b > best_brand_score:
        best_brand_score, best_brand_ncomp = r_b, n_comp

print(f"\nBest n_comp for memorability:       {best_mem_ncomp} (r={best_mem_score:.4f})")
print(f"Best n_comp for brand memorability: {best_brand_ncomp} (r={best_brand_score:.4f})")


# ── 2. Final cross-val with per-target optimal configs ────────────────────────
print("\n" + "="*60)
print("FINAL CV RESULTS — per-target optimal configs")

# Separate SVDs for each target
svd_mem   = TruncatedSVD(n_components=best_mem_ncomp,   random_state=42)
svd_brand = TruncatedSVD(n_components=best_brand_ncomp, random_state=42)

X_lsa_mem   = svd_mem.fit_transform(X_tfidf_v2)
X_lsa_brand = svd_brand.fit_transform(X_tfidf_v2)

feats_mem   = {f'lsa_{i}': X_lsa_mem[:,   i] for i in range(best_mem_ncomp)}
feats_brand = {f'lsa_{i}': X_lsa_brand[:, i] for i in range(best_brand_ncomp)}

r_m, _, final_preds_mem,   _ = rank_aggregate_cv(feats_mem,   y_mem, y_brand, df)
_, r_b, _, final_preds_brand = rank_aggregate_cv(feats_brand, y_mem, y_brand, df)
print(f"  Memorability Spearman r:       {r_m:.4f}")
print(f"  Brand Memorability Spearman r: {r_b:.4f}")


# ── 3. Per-fold final breakdown ───────────────────────────────────────────────
print("\nPer-fold breakdown:")
print(f"  {'Fold':<6} {'n':>4}  {'Mem r':>7}  {'Brand r':>8}  Channels")
print("  " + "-"*65)

all_pm = np.zeros(len(df))
all_pb = np.zeros(len(df))

for fold_id in range(5):
    val_mask   = (df['fold'] == fold_id).values
    train_mask = ~val_mask

    agg_m = np.zeros(val_mask.sum())
    agg_b = np.zeros(val_mask.sum())

    for j in range(best_mem_ncomp):
        col_tr, col_val = X_lsa_mem[train_mask, j], X_lsa_mem[val_mask, j]
        r, _ = spearmanr(col_tr, y_mem[train_mask])
        if abs(r) > 0.03:
            agg_m += np.sign(r) * rankdata(col_val).astype(float)

    for j in range(best_brand_ncomp):
        col_tr, col_val = X_lsa_brand[train_mask, j], X_lsa_brand[val_mask, j]
        r, _ = spearmanr(col_tr, y_brand[train_mask])
        if abs(r) > 0.03:
            agg_b += np.sign(r) * rankdata(col_val).astype(float)

    all_pm[val_mask] = agg_m
    all_pb[val_mask] = agg_b

    r_m_f = spearmanr(y_mem[val_mask],   agg_m).correlation
    r_b_f = spearmanr(y_brand[val_mask], agg_b).correlation
    chs   = ', '.join(df[val_mask]['channelName'].unique()[:2])
    print(f"  {fold_id:<6} {val_mask.sum():>4}  {r_m_f:>7.4f}  {r_b_f:>8.4f}  {chs}...")


# ── 4. Train FINAL models on ALL 339 videos ───────────────────────────────────
# For test set prediction we need:
# (a) TF-IDF fitted on all train transcripts
# (b) SVD fitted on all train TF-IDF
# (c) Direction of each LSA dim w.r.t. target on all train videos
# Then apply to test transcripts

print("\n" + "="*60)
print("FITTING FINAL MODELS ON ALL 339 TRAINING VIDEOS")

# Refit TF-IDF on all train data (already done as tfidf_v2)
# Refit SVD with optimal n_comp per target
final_svd_mem   = TruncatedSVD(n_components=best_mem_ncomp,   random_state=42)
final_svd_brand = TruncatedSVD(n_components=best_brand_ncomp, random_state=42)

X_train_lsa_mem   = final_svd_mem.fit_transform(X_tfidf_v2)
X_train_lsa_brand = final_svd_brand.fit_transform(X_tfidf_v2)

# Compute direction of each dim on full training set
mem_directions   = []
brand_directions = []
mem_dims_used    = []
brand_dims_used  = []

for j in range(best_mem_ncomp):
    r, _ = spearmanr(X_train_lsa_mem[:, j], y_mem)
    if abs(r) > 0.03:
        mem_directions.append(np.sign(r))
        mem_dims_used.append(j)

for j in range(best_brand_ncomp):
    r, _ = spearmanr(X_train_lsa_brand[:, j], y_brand)
    if abs(r) > 0.03:
        brand_directions.append(np.sign(r))
        brand_dims_used.append(j)

print(f"  Dims used for memorability:       {len(mem_dims_used)}/{best_mem_ncomp}")
print(f"  Dims used for brand memorability: {len(brand_dims_used)}/{best_brand_ncomp}")


# ── 5. Prediction function for test set ───────────────────────────────────────
def predict_test_videos(test_transcripts_dict, test_video_ids):
    """
    Given a dict of {video_id: transcript_text}, returns predictions.
    
    Parameters:
        test_transcripts_dict: dict {video_id: str}
        test_video_ids: list of video IDs in order
    
    Returns:
        pred_mem, pred_brand: arrays of shape (n_test,)
    """
    # Transform test transcripts with TRAIN-fitted TF-IDF
    test_corpus = [test_transcripts_dict.get(vid, '') for vid in test_video_ids]
    X_test_tfidf = tfidf_v2.transform(test_corpus)  # uses train vocab

    # Project into LSA space
    X_test_lsa_mem   = final_svd_mem.transform(X_test_tfidf)
    X_test_lsa_brand = final_svd_brand.transform(X_test_tfidf)

    # Rank aggregation: rank WITHIN test set, apply train-computed directions
    # Note: for a proper submission we rank within the test set
    pred_mem   = np.zeros(len(test_video_ids))
    pred_brand = np.zeros(len(test_video_ids))

    for j, sign in zip(mem_dims_used, mem_directions):
        pred_mem += sign * rankdata(X_test_lsa_mem[:, j]).astype(float)

    for j, sign in zip(brand_dims_used, brand_directions):
        pred_brand += sign * rankdata(X_test_lsa_brand[:, j]).astype(float)

    # Normalize to [0, 1] range to match score distribution
    def normalize(arr):
        mn, mx = arr.min(), arr.max()
        if mx == mn: return np.full_like(arr, 0.5)
        return (arr - mn) / (mx - mn)

    # Scale to match train score range
    pred_mem_norm   = normalize(pred_mem)
    pred_brand_norm = normalize(pred_brand)

    # Map to train score range
    pred_mem_scaled   = (pred_mem_norm *
                         (y_mem.max()   - y_mem.min())   + y_mem.min())
    pred_brand_scaled = (pred_brand_norm *
                         (y_brand.max() - y_brand.min()) + y_brand.min())

    return pred_mem_scaled, pred_brand_scaled


# ── 6. Validate predict_test_videos on train data (sanity check) ─────────────
print("\nSanity check — applying predict_test_videos to train set:")
train_transcript_dict = {vid: transcripts[vid] for vid in df['id']}
pm_check, pb_check = predict_test_videos(train_transcript_dict,
                                          df['id'].tolist())
r_m_check = spearmanr(y_mem,   pm_check).correlation
r_b_check = spearmanr(y_brand, pb_check).correlation
print(f"  Mem r (train, expected ~1.0):   {r_m_check:.4f}")
print(f"  Brand r (train, expected ~1.0): {r_b_check:.4f}")
print("  (High values confirm pipeline works correctly)")


# ── 7. Save everything needed for test prediction ─────────────────────────────
import pickle

pipeline = {
    'tfidf':              tfidf_v2,
    'svd_mem':            final_svd_mem,
    'svd_brand':          final_svd_brand,
    'mem_dims_used':      mem_dims_used,
    'mem_directions':     mem_directions,
    'brand_dims_used':    brand_dims_used,
    'brand_directions':   brand_directions,
    'y_mem_min':          float(y_mem.min()),
    'y_mem_max':          float(y_mem.max()),
    'y_brand_min':        float(y_brand.min()),
    'y_brand_max':        float(y_brand.max()),
    'best_mem_ncomp':     best_mem_ncomp,
    'best_brand_ncomp':   best_brand_ncomp,
    'cv_mem_spearman':    r_m,
    'cv_brand_spearman':  r_b,
}

with open('memorability_pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

print("\n" + "="*60)
print("Pipeline saved to memorability_pipeline.pkl")
print("\nFINAL SUMMARY:")
print(f"  CV Memorability Spearman r:       {r_m:.4f}")
print(f"  CV Brand Memorability Spearman r: {r_b:.4f}")
print(f"  Method: LSA({best_mem_ncomp}/{best_brand_ncomp}) + rank aggregation")
print(f"  Features: TF-IDF on STT transcripts only")
print(f"  No model parameters fitted (zero overfitting risk)")
print("\nTo predict test set, load the pipeline and call:")
print("  pred_mem, pred_brand = predict_test_videos(test_dict, test_ids)")

FINDING OPTIMAL n_comp PER TARGET
n_comp        Mem r  Brand r
------------------------------
50           0.1702   0.1609
80           0.1835   0.0579
100          0.1859   0.1729
120          0.1141   0.1008
150          0.2018   0.0386
200          0.1006   0.0871

Best n_comp for memorability:       150 (r=0.2018)
Best n_comp for brand memorability: 100 (r=0.1729)

FINAL CV RESULTS — per-target optimal configs
  Memorability Spearman r:       0.2018
  Brand Memorability Spearman r: 0.1729

Per-fold breakdown:
  Fold      n    Mem r   Brand r  Channels
  -----------------------------------------------------------------
  0        79   0.0879    0.0123  Goldman Sachs...
  1        65   0.2065   -0.0130  Legal & General, UBS...
  2        65  -0.0130    0.2508  jpmorgan, Legal & General Investment Management - LGIM...
  3        65   0.2122    0.3028  Allianz, BNP Paribas...
  4        65  -0.0080    0.0300  Aviva, Amundi...

FITTING FINAL MODELS ON ALL 339 TRAINING VIDEOS
  Dims used